# Generate the 15M-transaction synthetic dataset
Runs `ccc.sim.export` on Colab. Needs roughly 2 GB RAM and 2 GB disk; see the README for measured numbers. The planted-truth table is not exported.

In [ ]:
REPO_URL = "https://github.com/evangoh122/credit-card-case"
BRANCH = "main"  # change to the PR branch only if it is pushed
OUT_DIR = "/content/ccc-15m"  # or "/content/drive/MyDrive/ccc-15m" after mounting Drive
SEED = 1
TARGET_TRANSACTIONS = 15_000_000
HF_REPO_ID = "YOUR-USER/YOUR-DATASET"  # fill in before running the optional upload cell

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
!command -v uv >/dev/null 2>&1 || curl -LsSf https://astral.sh/uv/install.sh | sh
!rm -rf /content/ccc && git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/ccc
# uv manages the environment; the export only needs these packages.
EXPORT_DEPS = [
    "--with",
    "polars",
    "--with",
    "duckdb",
    "--with",
    "numpy",
    "--with",
    "scipy",
    "--with",
    "faker",
    "--with",
    "huggingface-hub",
]

In [ ]:
import os, sys

os.environ["PATH"] = f"{os.environ['HOME']}/.local/bin:" + os.environ["PATH"]
print(sys.version)
os.environ["PYTHONPATH"] = "/content/ccc/src"
!cd /content/ccc && uv run --no-project {" ".join(EXPORT_DEPS)} python -m ccc.sim.export --target-transactions {TARGET_TRANSACTIONS} --seed {SEED} --out {OUT_DIR}

In [ ]:
import os, subprocess

check = """
import os
import duckdb

out = os.environ["OUT_DIR"]
print(open(f"{out}/manifest.json").read())
for name in ("fact_transactions", "fact_events"):
    print(name, duckdb.sql(f"SELECT count(*) FROM read_parquet('{out}/{name}/*/*.parquet')").fetchall()[0][0])
print("dim_customer", duckdb.sql(f"SELECT count(*) FROM read_parquet('{out}/dim_customer.parquet')").fetchall()[0][0])
"""
subprocess.run(
    ["uv", "run", "--no-project", *EXPORT_DEPS, "python", "-c", check],
    env={**os.environ, "OUT_DIR": OUT_DIR},
    check=True,
)
!du -sh {OUT_DIR}

## Optional: upload to Hugging Face
The token comes from Colab Secrets (`HF_TOKEN`) or the environment and is never printed.

In [ ]:
import os, subprocess

try:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "add HF_TOKEN to Colab Secrets or the environment"
assert "YOUR-USER" not in HF_REPO_ID, "set HF_REPO_ID first"
env = {**os.environ, "HF_TOKEN": token, "PYTHONPATH": "/content/ccc/src"}
subprocess.run(
    [
        "uv",
        "run",
        "--no-project",
        *EXPORT_DEPS,
        "python",
        "-m",
        "ccc.sim.upload",
        "--dir",
        OUT_DIR,
        "--repo-id",
        HF_REPO_ID,
        "--create",
    ],
    env=env,
    check=True,
)